Calculate Statistical Summaries

In [0]:
spark.sql("USE CATALOG ecommerce_uc")
spark.sql("USE SCHEMA bronze_layer")
events = spark.table("events")
events.describe().show(3)


+-------+----------+--------------------+--------------------+-------------+--------+-----------------+--------------------+------------+
|summary|event_type|          product_id|         category_id|category_code|   brand|            price|             user_id|user_session|
+-------+----------+--------------------+--------------------+-------------+--------+-----------------+--------------------+------------+
|  count|  67501979|            67501979|            67501979|     45603808|58283744|         67501979|            67501979|    67501969|
|   mean|      NULL|1.2514064889882294E7|2.057897644306974E18|         NULL|     NaN|292.4593165648948| 5.386397456296759E8|        NULL|
| stddev|      NULL| 1.725741362984618E7|2.012549032884273...|         NULL|     NaN|355.6744995860673|2.2885161051522106E7|        NULL|
+-------+----------+--------------------+--------------------+-------------+--------+-----------------+--------------------+------------+
only showing top 3 rows


In [0]:
events.describe("price").show(3)


+-------+------------------+
|summary|             price|
+-------+------------------+
|  count|          67501979|
|   mean| 292.4593165647889|
| stddev|355.67449958606727|
+-------+------------------+
only showing top 3 rows


In [0]:
from pyspark.sql import functions as F

events.select(
    F.mean("price").alias("mean_price"),
    F.stddev("price").alias("std_price"),
    F.min("price").alias("min_price"),
    F.max("price").alias("max_price")
).show(3)


+------------------+------------------+---------+---------+
|        mean_price|         std_price|min_price|max_price|
+------------------+------------------+---------+---------+
|292.45931656462966|355.67449958606727|      0.0|  2574.07|
+------------------+------------------+---------+---------+



## Hypothesis Testing (Weekday vs Weekend)

In [0]:
from pyspark.sql import functions as F

events_flagged = events.withColumn(
    "is_weekend",
    F.dayofweek("event_time").isin([1, 7])  # Sunday=1, Saturday=7
)


In [0]:
events_flagged.groupBy("is_weekend", "event_type") .count().orderBy("is_weekend", "event_type").show()


+----------+----------+--------+
|is_weekend|event_type|   count|
+----------+----------+--------+
|     false|      cart| 1799242|
|     false|  purchase|  500258|
|     false|      view|40453993|
|      true|      cart| 1229688|
|      true|  purchase|  416681|
|      true|      view|23102117|
+----------+----------+--------+



## Identify Correlations

In [0]:


from pyspark.sql import functions as F

conversion_df = (
    events
    .groupBy("product_id")
    .agg(
        F.sum(F.when(F.col("event_type") == "purchase", 1).otherwise(0)).alias("purchases"),
        F.sum(F.when(F.col("event_type") == "view", 1).otherwise(0)).alias("views"),
        F.avg("price").alias("avg_price")
    )
    .withColumn(
        "conversion_rate",
        F.col("purchases") / F.col("views")
    )
    .filter(F.col("views") > 50)  # remove noise
)


In [0]:
conversion_df.printSchema()
conversion_df.show(5)


root
 |-- product_id: integer (nullable = true)
 |-- purchases: long (nullable = true)
 |-- views: long (nullable = true)
 |-- avg_price: double (nullable = true)
 |-- conversion_rate: double (nullable = true)

+----------+---------+------+------------------+--------------------+
|product_id|purchases| views|         avg_price|     conversion_rate|
+----------+---------+------+------------------+--------------------+
|   1005159|     2600|106884|202.31828069761048|0.024325436922270873|
|   6902812|        4|   473| 81.33617408906888|0.008456659619450317|
|   7004004|       12|   700|128.68000000000006|0.017142857142857144|
|   8500290|       40|  1516| 262.0436842105265|0.026385224274406333|
|  23700185|        0|    80|30.022499999999997|                 0.0|
+----------+---------+------+------------------+--------------------+
only showing top 5 rows


In [0]:
conversion_df.stat.corr("avg_price", "conversion_rate")



-0.13088483306753926

read correlation

Near +1 → strong positive

Near -1 → strong negative

Near 0 → weak or no relationship

In [0]:
from pyspark.sql import Window

features_df = events.withColumn("day_of_week", F.dayofweek("event_time")) \
    .withColumn("is_weekend",
        F.dayofweek("event_time").isin([1, 7])
    )


In [0]:
features_df = features_df.withColumn(
    "price_log",
    F.log(F.col("price") + 1)
)


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

window = Window.partitionBy("user_id").orderBy("event_time")

features_df = events.withColumn(
    "time_since_first_event",
    F.unix_timestamp("event_time") -
    F.unix_timestamp(F.first("event_time").over(window))
)




In [0]:
features_df.printSchema()


root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)
 |-- time_since_first_event: long (nullable = true)



In [0]:
features_df.show(2)


+-------------------+----------+----------+-------------------+-------------+-------+-----+--------+--------------------+----------------------+
|         event_time|event_type|product_id|        category_id|category_code|  brand|price| user_id|        user_session|time_since_first_event|
+-------------------+----------+----------+-------------------+-------------+-------+-----+--------+--------------------+----------------------+
|2019-11-08 07:44:45|      view|  16400235|2053013558249128509|         NULL|bergner|66.35|81255481|eafa8f8e-7e45-4f6...|                     0|
|2019-11-21 14:11:26|      view|  16400235|2053013558249128509|         NULL|bergner|66.14|81255481|e9ff380a-a553-49d...|               1146401|
+-------------------+----------+----------+-------------------+-------------+-------+-----+--------+--------------------+----------------------+
only showing top 2 rows
